In [1]:
# Imports

library(dslabs)
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.6.0
✔ ggplot2   3.5.1     ✔ tibble    3.3.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


For serious data analysis, not vectors but data frames are the unit of analysis. So-called "tidy" data frames are used with an interrelated set of packages known as the "tidyverse" which was loaded all with the single line `library(tidyverse)` above.

The tidyverse approach to doing things will be introduced in the text but first there will be a little hands-on experience without too much theory. Some of the packages in the tidyverse include:

* dplyr (used for manipulating tibbles)
* purrr (extends functional programming for the tidyverse)
* ggplot2 (declarative graphing package based on Leland Wilkinson’s [*Grammar of Graphics*](https://en.wikipedia.org/wiki/Wilkinson%27s_Grammar_of_Graphics)
* readr (provides fast and easy reading of many forms of tabular data into R tibbles)

Tidy data must meet the following three criteria:

1. Each row represents one observation.
2. Each column represents one of the variables present in each observation.
3. Each cell contains only one value.

The `murders` dataset is an example of tidy data: 

In [2]:
head(murders)

,state,abb,region,population,total
,<chr>,<chr>,<fct>,<dbl>,<dbl>
1,Alabama,AL,South,4779736,135
2,Alaska,AK,West,710231,19
3,Arizona,AZ,West,6392017,232
4,Arkansas,AR,South,2915918,93
5,California,CA,West,37253956,1257
6,Colorado,CO,West,5029196,65


Here's another example of a tidy dataset, showing fertility rates for two countries over three years:

```
#>       country year fertility
#> 1     Germany 1960      2.41
#> 2 South Korea 1960      6.16
#> 3     Germany 1961      2.44
#> 4 South Korea 1961      5.99
#> 5     Germany 1962      2.47
#> 6 South Korea 1962      5.79
```

The original data were in the sort of "wide" format that one often sees for human convenience:

```
#>       country 1960 1961 1962
#> 1     Germany 2.41 2.44 2.47
#> 2 South Korea 6.16 5.99 5.79
```

Here there are multiple observations on each row as the `year` variable is distributed across multiple columns, violating tidy data principles. Typically tidy data will be in a "long" format, though less frequently it may be necessary to convert single observations split across multiple rows to a wide format. In any case, tidy data principles require an initial investment of discipline but, once this is done, the user will be freed to think about more important aspects of the analysis than the format of the data. Now for some exercises:

**Exercise 1**: Examine the built-in dataset `co2`. Which of the following is true:

1. `co2` is tidy data: it has one year for each row.
2. `co2` is not tidy: we need at least one column with a character vector.
3. `co2` is not tidy: it is a matrix instead of a data frame.
4. `co2` is not tidy: to be tidy we would have to wrangle it to have three columns (year, month and value), then each `co2` observation would have a row.

#4

**Exercise 2**: Examine the built-in dataset `ChickWeight`. Which of the following is true:

1. `ChickWeight` is not tidy: each chick has more than one row.
2. `ChickWeight` is tidy: each observation (a weight) is represented by one row. The chick from which this measurement came is one of the variables.
3. `ChickWeight` is not tidy: we are missing the year column.
4. `ChickWeight` is tidy: it is stored in a data frame.

#2

**Exercise 3**: Examine the built-in dataset `BOD`. Which of the following is true:

1. `BOD` is not tidy: it only has six rows.
2. `BOD` is not tidy: the first column is just an index.
3. `BOD` is tidy: each row is an observation with two values (time and demand)
4. `BOD` is tidy: all small datasets are tidy by definition.

#3

**Exercise 4**: Which of the following built-in datasets is tidy (you can pick more than one):

1. `BJsales`
2. `EuStockMarkets`
3. `DNase`
4. `Formaldehyde`
5. `Orange`
6. `UCBAdmissions`

#1, #3, #4, #5

The next section concerns the basics of dplyr, which provides staple data manipulation operations for with R data frames and tibbles. Among the operations provided by dplyr are:

* `mutate()`, which transforms columns that can then be added to existing data if desired.
* `filter()`, which returns row-based subsets of the data.
* `select()`, which subsets by columns rather than rows.

Here's our first example using `mutate()` to add the per capita (per 100,000) homicide rate to `murders`. Note that merely running `mutate()` will *not* change the data structure in-place; rather the mutated copy must be assigned to the variable `murders`. Notice also that `mutate()`, like other dplyr "verbs", uses the [magic of non-standard evaluation in R](https://dplyr.tidyverse.org/articles/programming.html) to avoid having to spell everything out like one would have to with pandas in Python:

In [3]:
murders <- mutate(murders, rate = total / population * 10^5)
head(murders)

,state,abb,region,population,total,rate
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
1,Alabama,AL,South,4779736,135,2.824424
2,Alaska,AK,West,710231,19,2.675186
3,Arizona,AZ,West,6392017,232,3.629527
4,Arkansas,AR,South,2915918,93,3.189390
5,California,CA,West,37253956,1257,3.374138
6,Colorado,CO,West,5029196,65,1.292453


`filter()` does the aforementioned row-based subsetting. Here's an example with one Boolean condition and notice the same non-standard evaluation:

In [4]:
filter(murders, rate <= 0.71)

state,abb,region,population,total,rate
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
Hawaii,HI,West,1360301,7,0.5145920
Iowa,IA,North Central,3046355,21,0.6893484
New Hampshire,NH,Northeast,1316470,5,0.3798036
North Dakota,ND,North Central,672591,4,0.5947151
Vermont,VT,Northeast,625741,2,0.3196211


Now for column-wise subsetting: `murders` has only six columns, but many datasets (especially before tidying) may have many more columns. Both for the purposes of human ease of use and to make machine analysis simpler, faster, and very possibly more effective (due to [including only the most important features](https://towardsdatascience.com/how-to-mitigate-overfitting-with-feature-selection-164897c0c3db/)), `select()` can be used to subset columns of a data frame or tibble. Here's a maybe not very useful but simple example based on `murders`:

In [5]:
new_df <- select(murders, state, region, rate)
filter(new_df, rate <= 0.71)

state,region,rate
<chr>,<fct>,<dbl>
Hawaii,West,0.5145920
Iowa,North Central,0.6893484
New Hampshire,Northeast,0.3798036
North Dakota,North Central,0.5947151
Vermont,Northeast,0.3196211


The Tidyverse package tidyselect offers a number of functions to select columns based on their content and not just by their full literal names. For example, here are the `numeric` columns of `murders` only:

In [6]:
new_df <- select(murders, where(is.numeric))
head(new_df)

,population,total,rate
,<dbl>,<dbl>,<dbl>
1,4779736,135,2.824424
2,710231,19,2.675186
3,6392017,232,3.629527
4,2915918,93,3.189390
5,37253956,1257,3.374138
6,5029196,65,1.292453


Here are a few other tidyselect functions and their descriptions:

* `starts_with()` (starts with a literal prefix).
* `ends_with()` (ends with a literal suffix).
* `contains()` (contains a literal string anywhere).
* `matches()` (matches a regular expression; there are nuances here such as the ability to use Tidyverse [stringr](https://stringr.tidyverse.org/articles/regular-expressions.html) objects).
* `num_range()` (matches strings like `wk01`, `wk02`, `wk03` etc.; [tidyselect documentation shows example use](https://tidyselect.r-lib.org/reference/starts_with.html)).

All of the above functions have an `ignore.case` parameter whose default argument is `TRUE` except for `num_range()`, which appears to be case-sensitive always. Anyway, here's a quick example of `starts_with()`:

In [7]:
new_df <- select(murders, starts_with('r'))
head(new_df)

,region,rate
,<fct>,<dbl>
1,South,2.824424
2,West,2.675186
3,West,3.629527
4,South,3.189390
5,West,3.374138
6,West,1.292453


Returning to the aforementioned `mutate()`, here are a few more examples, such as a $log_{10}(x)$ transformation:

In [8]:
head(mutate(murders, population = log10(population)))

,state,abb,region,population,total,rate
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
1,Alabama,AL,South,6.679404,135,2.824424
2,Alaska,AK,West,5.851400,19,2.675186
3,Arizona,AZ,West,6.805638,232,3.629527
4,Arkansas,AR,South,6.464775,93,3.189390
5,California,CA,West,7.571172,1257,3.374138
6,Colorado,CO,West,6.701499,65,1.292453


The `across()` function allows several ways to use `mutate()` on multiple columns. Here it is used with two literals:

In [9]:
head(mutate(murders, across(c(population, total), log10)))

,state,abb,region,population,total,rate
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
1,Alabama,AL,South,6.679404,2.130334,2.824424
2,Alaska,AK,West,5.851400,1.278754,2.675186
3,Arizona,AZ,West,6.805638,2.365488,3.629527
4,Arkansas,AR,South,6.464775,1.968483,3.189390
5,California,CA,West,7.571172,3.099335,3.374138
6,Colorado,CO,West,6.701499,1.812913,1.292453


Predicate functions can also be used with `across()`. Here, all `numeric` variables are transformed with `log10()`:

In [10]:
head(mutate(murders, across(where(is.numeric), log10)))

,state,abb,region,population,total,rate
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
1,Alabama,AL,South,6.679404,2.130334,0.4509299
2,Alaska,AK,West,5.851400,1.278754,0.4273540
3,Arizona,AZ,West,6.805638,2.365488,0.5598501
4,Arkansas,AR,South,6.464775,1.968483,0.5037076
5,California,CA,West,7.571172,3.099335,0.5281629
6,Colorado,CO,West,6.701499,1.812913,0.1114148


And here, `character` variables are lowercased:

In [11]:
head(mutate(murders, across(where(is.character), tolower)))

,state,abb,region,population,total,rate
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
1,alabama,al,South,4779736,135,2.824424
2,alaska,ak,West,710231,19,2.675186
3,arizona,az,West,6392017,232,3.629527
4,arkansas,ar,South,2915918,93,3.189390
5,california,ca,West,37253956,1257,3.374138
6,colorado,co,West,5029196,65,1.292453


Now for a few more exercises.

**Exercise 5**: You can add columns using the dplyr function `mutate()`. This function is aware of the column names and inside the function you can call them unquoted:

```r
murders <- mutate(murders, population_in_millions = population/10^6)
```

We can write `population` rather than `murders$population`. The function `mutate()` knows we are grabbing columns from `murders`.

Use the function `mutate()` to add a `murders` column named `rate` with the per 100,000 murder rate \[similar to\] the example code above. Make sure you redefine `murders` as done in the example code above (murders <- \[your code\]) so we can keep using this variable.

In [12]:
murders <- mutate(murders, rate = total / population * 10^5)
head(murders)

,state,abb,region,population,total,rate
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>
1,Alabama,AL,South,4779736,135,2.824424
2,Alaska,AK,West,710231,19,2.675186
3,Arizona,AZ,West,6392017,232,3.629527
4,Arkansas,AR,South,2915918,93,3.189390
5,California,CA,West,37253956,1257,3.374138
6,Colorado,CO,West,5029196,65,1.292453


**Exercise 6**: If `rank(x)` gives you the ranks of `x` from lowest to highest, `rank(-x)` gives you the ranks from highest to lowest. Use the function `mutate()` to add a column `rank` containing the rank of murder rate from highest to lowest. Make sure you redefine `murders` so we can keep using this variable.

In [13]:
murders <- mutate(murders, rank = rank(-rate))
head(murders)

,state,abb,region,population,total,rate,rank
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
1,Alabama,AL,South,4779736,135,2.824424,23
2,Alaska,AK,West,710231,19,2.675186,27
3,Arizona,AZ,West,6392017,232,3.629527,10
4,Arkansas,AR,South,2915918,93,3.189390,17
5,California,CA,West,37253956,1257,3.374138,14
6,Colorado,CO,West,5029196,65,1.292453,38


N.b.: `arrange()` will come up later but it can absolutely be used here:

In [14]:
arrange(murders, rank)

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
District of Columbia,DC,South,601723,99,16.4527532,1
Louisiana,LA,South,4533372,351,7.7425810,2
Missouri,MO,North Central,5988927,321,5.3598917,3
Maryland,MD,South,5773552,293,5.0748655,4
South Carolina,SC,South,4625364,207,4.4753235,5
Delaware,DE,South,897934,38,4.2319369,6
Michigan,MI,North Central,9883640,413,4.1786225,7
Mississippi,MS,South,2967297,120,4.0440846,8
Georgia,GA,South,9920000,376,3.7903226,9


**Exercise 7**: With dplyr, we can use `select()` to show only certain columns. For example, with this code we would only show the states and population sizes:

```r
select(murders, state, population)
```

\[The following also works:\]

```r
select(murders, c(state, population))
```

Use `select()` to show the state names and abbreviations in `murders`. Do not redefine `murders`, just show the results.

In [15]:
head(select(murders, c(state, abb)))

,state,abb
,<chr>,<chr>
1,Alabama,AL
2,Alaska,AK
3,Arizona,AZ
4,Arkansas,AR
5,California,CA
6,Colorado,CO


**Exercise 8**: The dplyr function `filter()` is used to choose specific rows of the data frame to keep. Unlike `select()`, which is for columns, `filter()` is for rows. For example, you can show just the New York row like this:

```r
filter(murders, state == 'New York')
```

You can use other `logical` vectors to filter rows.

Use `filter()` to show the top 5 states with the highest murder rates. From here on, do not change the `murders` dataset, just show the result. Remember that you can `filter()` based on the `rank` column. \[I will be sorting using `arrange()` again.\]

In [16]:
filter(murders, rank <= 5) |> arrange(rank)

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
District of Columbia,DC,South,601723,99,16.452753,1
Louisiana,LA,South,4533372,351,7.742581,2
Missouri,MO,North Central,5988927,321,5.359892,3
Maryland,MD,South,5773552,293,5.074866,4
South Carolina,SC,South,4625364,207,4.475323,5


**Exercise 9**: We can remove rows using the `!=` operator. For example, to remove Florida, we would do this:

```r
no_florida <- filter(murders, state != 'Florida')
```

Create a new data frame called `no_south` that removes states from the South region. How many states are in this category? You can use the function `nrow()` for this.

In [18]:
no_south <- filter(murders, region != 'South')
no_south

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
Alaska,AK,West,710231,19,2.6751860,27
Arizona,AZ,West,6392017,232,3.6295273,10
California,CA,West,37253956,1257,3.3741383,14
Colorado,CO,West,5029196,65,1.2924531,38
Connecticut,CT,Northeast,3574097,97,2.7139722,25
Hawaii,HI,West,1360301,7,0.5145920,49
Idaho,ID,West,1567582,12,0.7655102,46
Illinois,IL,North Central,12830632,364,2.8369608,22
Indiana,IN,North Central,6483802,142,2.1900730,31


In [19]:
nrow(no_south)

[1] 34

**Exercise 10**: We can also use `%in%` to filter with dplyr. You can therefore see the data from New York and Texas like this:

```r
filter(murders, state %in% c('New York', 'Texas'))
```

Create a new data frame called `murders_nw` with only the states from the Northeast and the West. How many states are in this category?

In [22]:
murders_nw <- filter(murders, region %in% c('Northeast', 'West'))
murders_nw

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
Alaska,AK,West,710231,19,2.6751860,27
Arizona,AZ,West,6392017,232,3.6295273,10
California,CA,West,37253956,1257,3.3741383,14
Colorado,CO,West,5029196,65,1.2924531,38
Connecticut,CT,Northeast,3574097,97,2.7139722,25
Hawaii,HI,West,1360301,7,0.5145920,49
Idaho,ID,West,1567582,12,0.7655102,46
Maine,ME,Northeast,1328361,11,0.8280881,44
Massachusetts,MA,Northeast,6547629,118,1.8021791,32


In [23]:
nrow(murders_nw)

[1] 22

**Exercise 11**: Suppose you want to live in the Northeast or West and want the murder rate to be less than 1. We want to see the data for the states satisfying these options. Note that you can use logical operators like `&` with `filter()`. Here is an example in which we filter to keep only small states in the Northeast region.

```r
filter(murders, population < 5000000 & region == 'Northeast')
```

Make sure `murders` has been defined with `rate` and `rank` and still has all states. Create a table called `my_states` that contains rows for states satisfying both the conditions: it is in the Northeast or West and the murder rate is less than 1. Use `select()` to show only the state name, the rate, and the rank.

In [28]:
my_states <- filter(murders, region %in% c('Northeast', 'West') & rate < 1)
my_states

state,abb,region,population,total,rate,rank
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>
Hawaii,HI,West,1360301,7,0.5145920,49
Idaho,ID,West,1567582,12,0.7655102,46
Maine,ME,Northeast,1328361,11,0.8280881,44
New Hampshire,NH,Northeast,1316470,5,0.3798036,50
Oregon,OR,West,3831074,36,0.9396843,42
Utah,UT,West,2763885,22,0.7959810,45
Vermont,VT,Northeast,625741,2,0.3196211,51
Wyoming,WY,West,563626,5,0.8871131,43
